# Notebook for Collisions

In [17]:
import sympy as sp
import numpy as np

In [28]:
## Setting up the problem

# velocity in cartesian coordinates
vc = sp.Array(sp.symbols('v_0 v_1 v_2'))

# diffusion coefficients
kpar, kperp = sp.symbols(R'\kappa_\parallel \kappa_\perp')

# diffusion tensor in cartesian coordinates
dc = kperp * (sp.Array(np.eye(3, dtype=int))*sum(v**2 for v in vc) - sp.tensorproduct(vc, vc)) + kpar * sp.tensorproduct(vc, vc)

dc

[[\kappa_\parallel*v_0**2 + \kappa_\perp*(v_1**2 + v_2**2), \kappa_\parallel*v_0*v_1 - \kappa_\perp*v_0*v_1, \kappa_\parallel*v_0*v_2 - \kappa_\perp*v_0*v_2], [\kappa_\parallel*v_0*v_1 - \kappa_\perp*v_0*v_1, \kappa_\parallel*v_1**2 + \kappa_\perp*(v_0**2 + v_2**2), \kappa_\parallel*v_1*v_2 - \kappa_\perp*v_1*v_2], [\kappa_\parallel*v_0*v_2 - \kappa_\perp*v_0*v_2, \kappa_\parallel*v_1*v_2 - \kappa_\perp*v_1*v_2, \kappa_\parallel*v_2**2 + \kappa_\perp*(v_0**2 + v_1**2)]]

In [51]:
# Set up spherical coordinates
# magnitude of velocity
vmag = sp.sqrt(sum(v**2 for v in vc))
# cosine of the polar angle
xi = vc[0] / vmag
# azimuthal angle
phi = sp.atan2(vc[1], vc[2])

vsph = sp.Array([vmag, xi])

jacobian = sp.Array([[sp.diff(var, v) for v in vc] for var in vsph])

In [53]:
dsph = sp.Array([[sum(jacobian[i,k] * dc[k,l] * jacobian[j,l] for k in range(3) for l in range(3)) for i in range(2)] for j in range(2)])

In [54]:
sp.simplify(dsph)

[[\kappa_\parallel*(v_0**2 + v_1**2 + v_2**2), 0], [0, \kappa_\perp*(v_1**2 + v_2**2)/(v_0**2 + v_1**2 + v_2**2)]]

In [76]:
# check diffusion tensor in GC coordinates
p, xi = sp.symbols(R'p \xi')
m, B = sp.symbols('m B')
vpar = p * xi / m
mu = p**2 * (1 - xi**2) / (2 * m * B)
jacobian = sp.Array([[sp.diff(var, v) for v in [p, xi]] for var in [vpar, mu]])

dpar, dperp = sp.symbols(R'D_\parallel D_\perp')
# Lower block of the matrix in equation (49)
dsph = sp.Array([[dpar, 0], [0, (1-xi**2)*dperp/p**2]])

# Compute the transform from (p, xi) to (vpar, mu)
d1 = sp.Array([[sum(jacobian[i,k] * dsph[k,l] * jacobian[j,l] for k in range(2) for l in range(2)) for i in range(2)] for j in range(2)])

# Write the expression in eq (57)
K = p**2 / (2*m)
d2 = sp.Array([
    [dpar / m**2 + (dperp - dpar)/m**2 * mu * B / K,
     mu * vpar / (m * K) * (dpar - dperp)],
    [mu * vpar / (m * K) * (dpar - dperp),
     (K/B)**2 * 4 / p**2 * (1-xi**2) * (dpar*(1-xi**2) + dperp*xi**2)]
    ])

# Compare
sp.simplify(d1 - d2)

[[0, 0], [0, 0]]

In [77]:
jacobian

[[\xi/m, p/m], [p*(1 - \xi**2)/(B*m), -\xi*p**2/(B*m)]]